In [7]:
import os
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types 
from typing import Optional, Dict, Any

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.CRITICAL)


In [12]:
MODEL_GPT = "groq/openai/gpt-oss-120b"
llm = LiteLlm(model=MODEL_GPT)

print(
    llm.llm_client.completion(
        model=llm.model,
        messages=[
            {
                "role": "user",
                "content": "Are you ready?"
            }
        ],
        tools=[]
    )
)
print("\nGroq is ready for use.")

ModelResponse(id='chatcmpl-ef4d2101-04ea-4a41-b973-07ffb0c01aeb', created=1788624297, model='openai/gpt-oss-120b', object='chat.completion', system_fingerprint='fp_96d96a151c', choices=[Choices(finish_reason='stop', index=0, message=Message(content="Absolutely—I'm ready! How can I assist you today?", role='assistant', tool_calls=None, function_call=None, provider_specific_fields=None, reasoning='We need to respond according to the system instructions. The system says: "You are ChatGPT...". There\'s no disallowed content. User asks "Are you ready?" We respond affirmatively.'))], usage=Usage(completion_tokens=61, prompt_tokens=75, total_tokens=136, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=40, rejected_prediction_tokens=None, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=None, queue_time=0.308717444, prompt_time=0.030601355, completion_time=0.127531291, total_time=0.158132

In [19]:
from neo4j_for_adk import graphdb

In [20]:
neo4j_is_ready = graphdb.send_query("RETURN 'Neo4j is Ready!' as message")

print(neo4j_is_ready)

{'status': 'success', 'query_result': [{'message': 'Neo4j is Ready!'}]}


In [25]:
def say_hello(person_name: str) -> dict:
    """Formats a welcome message to a named person. 

    Args:
        person_name (str): the name of the person saying hello

    Returns:
        dict: A dictionary containing the results of the query.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes a 'query_result' key with an array of result rows.
              If 'error', includes an 'error_message' key.
    """
    return graphdb.send_query("RETURN 'Hello to you, ' + $person_name AS reply",
    {
        "person_name": person_name
    })

In [26]:
print(say_hello("Arshad"))

{'status': 'success', 'query_result': [{'reply': 'Hello to you, Arshad'}]}


# friendly_cypher_agent

In [27]:
hello_agent = Agent(
    name="hello_agent_v1",
    model=llm,
    description="Has friendly chats with a user.",
    instruction="""You are a helpful assistant, chatting with a user. 
                Be polite and friendly, introducing yourself and asking who the user is. 

                If the user provides their name, use the 'say_hello' tool to get a custom greeting.
                If the tool returns an error, inform the user politely. 
                If the tool is successful, present the reply.
                """,
    tools=[say_hello], 
)

print(f"Agent '{hello_agent.name}' created.")

Agent 'hello_agent_v1' created.


# Run the Agent

To run an agent, we need some additional components namely an execution environment and memory.

## Create the Runner and SessionService


Let's assume we have a single user talking to the agent in a single session. Let's create this user, the session and the runner:
* `SessionService`: Responsible for managing conversation history and state for different users and sessions. The `InMemorySessionService` is a simple implementation that stores everything in memory, suitable for testing and simple applications. It keeps track of the messages exchanged.  
* `Runner`: The engine that orchestrates the interaction flow. It takes user input, routes it to the appropriate agent, manages calls to the LLM and tools based on the agent's logic, handles session updates via the `SessionService`, and yields events representing the progress of the interaction.

In [30]:
app_name = hello_agent.name + "_app"
user_id = hello_agent.name + "_user"
session_id = hello_agent.name + "_session_01"
    
session_service = InMemorySessionService()
await session_service.create_session(
    app_name=app_name,
    user_id=user_id,
    session_id=session_id
)
    
runner = Runner(
    agent=hello_agent,
    app_name=app_name,
    session_service=session_service
)

In [34]:
user_message = "Hello, I'm Arshad"

print(f"\n>>> User Message: {user_message}")

content = types.Content(
    role="user",
    parts=[types.Part(text=user_message)]
)

final_response_text = "Agent did not produce a final response."

async for event in runner.run_async(
    user_id=user_id,
    session_id=session_id,
    new_message=content
):

    print("\n========== EVENT ==========")
    print("Author:", event.author)
    print("Final:", event.is_final_response())
    print("Content:", event.content)
    print("Actions:", event.actions)

    if event.is_final_response():

        # Check for text response
        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, "text") and part.text:
                    final_response_text = part.text

        break

print(f"\n<<< Agent Response: {final_response_text}")


>>> User Message: Hello, I'm Arshad

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


========== EVENT ==========
Author: hello_agent_v1
Final: True
Content: None
Actions: skip_summarization=None state_delta={} artifact_delta={} transfer_to_agent=None escalate=None requested_auth_configs={} requested_tool_confirmations={} compaction=None end_of_agent=None agent_state=None rewind_before_invocation_id=None route=None render_ui_widgets=None set_model_response=None

<<< Agent Response: Agent did not produce a final response.
